<a href="https://colab.research.google.com/github/anggaa0519/data-science-2026/blob/main/Pertemuan3_angga_anggieanie_250401020172.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tugas Pertemuan 3 — Data Cleaning: Missing Values, Outlier & Ekstraksi Data

| | |
|---|---|
| **Nama** | Angga Anggieanie |
| **NIM** | 250401020172 |
| **Kelas** | IF401 |
| **Mata Kuliah** | Pengantar Data Science |
| **Pertemuan** | 3 — Missing Values, Outlier & Ekstraksi Data |


In [44]:


import pandas as pd, numpy as np
from scipy.stats.mstats import winsorize

## 'STEP 0 — Load & eksplorasi awal'

df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/bahan/housing_dirty.csv')
print("Shape awal:", df.shape)
display(df.head())

print("\nInfo dataset:")
df.info()

print("\nStatistik deskriptif:")
display(df.describe(include="all"))

print("\nJumlah missing value:")
print(df.isnull().sum())

print("\nJumlah duplikat:")
print(df.duplicated().sum())

## `STEP 1 — Hapus Duplikat`
df.drop_duplicates(inplace=True)
print('Setelah hapus duplikat:', df.shape)

## `STEP 2 — Normalisasi String`
df["kota"] = df["kota"].str.strip().str.title()
df["kondisi"] = df["kondisi"].str.strip().str.lower()
display(df[["kota", "kondisi"]].head())

## 'STEP 3 — Imputasi Missing Values'
df['luas_m2'] = df['luas_m2'].fillna(df['luas_m2'].median())
df['harga_juta'] = df['harga_juta'].fillna(df['harga_juta'].median())
df['kamar'] = df['kamar'].fillna(df['kamar'].mode()[0])

print("Total missing value setelah imputasi:", df.isnull().sum().sum())
print(df.isnull().sum())

## 'STEP 4 — Tangani Outlier (IQR Fence)'
for col in ["harga_juta", "luas_m2", "tahun_bangun"]:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df[col] = df[col].clip(lower, upper)

    print(f"{col}")
    print(f"Batas bawah: {lower}")
    print(f"Batas atas : {upper}")
    print()

## 'STEP 5 — Validasi & Ekspor'
assert df.isnull().sum().sum() == 0, 'Masih ada missing!'
assert df.duplicated().sum() == 0, 'Masih ada duplikat!'

print('Shape akhir:', df.shape)
df.to_csv('housing_clean.csv', index=False)
print('Dataset bersih tersimpan!')

display(df.head())

## 'STEP 6 - Ekspor dataset bersih'
df.to_csv("housing_clean.csv", index=False)

print("Dataset bersih berhasil disimpan sebagai housing_clean.csv")

## 'STEP 7 - Akses REST API JSONPlaceholder'
import requests

url ="https://jsonplaceholder.typicode.com/users"
response = requests.get(url, timeout=10 )

if response.status_code == 200:
  data = response.json()
  df_api = pd.json_normalize(data, sep='_')

  display (df_api[["id", "name", "email", "address_city"]])
else:
  print("Error", response.status_code)

  ## 'STEP 8 - Simpan hasil API ke CSV'
  df_api.to_csv("jsonplaceholder_users.csv", index=False)

print("Data API berhasil disimpan sebagai jsonplaceholder_users.csv")

Shape awal: (130, 7)


,id,luas_m2,harga_juta,kota,kamar,tahun_bangun,kondisi
0,1,297.0,1084.0,jogja,2.0,2000,baik
1,2,254.0,761.0,Medan,NaN,1995,Bagus
2,3,249.7,895.0,Depok,NaN,1983,baik
3,4,49.7,178.0,YGY,5.0,2013,baik
4,5,133.4,424.0,Medan,5.0,2004,Sedang



Info dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 130 entries, 0 to 129
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            130 non-null    int64  
 1   luas_m2       112 non-null    float64
 2   harga_juta    113 non-null    float64
 3   kota          130 non-null    object 
 4   kamar         120 non-null    float64
 5   tahun_bangun  130 non-null    int64  
 6   kondisi       130 non-null    object 
dtypes: float64(3), int64(2), object(2)
memory usage: 7.2+ KB

Statistik deskriptif:


,id,luas_m2,harga_juta,kota,kamar,tahun_bangun,kondisi
count,130.000000,112.000000,1.130000e+02,130,120.000000,130.000000,130
unique,NaN,NaN,NaN,31,NaN,NaN,15
top,NaN,NaN,NaN,Bandung,NaN,NaN,baik
freq,NaN,NaN,NaN,14,NaN,NaN,45
mean,65.500000,267.627679,8.856325e+05,NaN,3.433333,2062.638462,NaN
std,37.671829,885.664181,9.407144e+06,NaN,1.776283,701.684043,NaN
min,1.000000,-50.000000,-5.000000e+02,NaN,1.000000,1890.000000,NaN
25%,33.250000,87.050000,3.450000e+02,NaN,2.000000,1991.250000,NaN
50%,65.500000,193.800000,6.550000e+02,NaN,4.000000,2002.000000,NaN
75%,97.750000,280.675000,9.550000e+02,NaN,5.000000,2011.750000,NaN



Jumlah missing value:
id               0
luas_m2         18
harga_juta      17
kota             0
kamar           10
tahun_bangun     0
kondisi          0
dtype: int64

Jumlah duplikat:
0
Setelah hapus duplikat: (130, 7)


...

Total missing value setelah imputasi: 0
id              0
luas_m2         0
harga_juta      0
kota            0
kamar           0
tahun_bangun    0
kondisi         0
dtype: int64
harga_juta
Batas bawah: -422.75
Batas atas : 1719.25

luas_m2
Batas bawah: -145.225
Batas atas : 512.9749999999999

tahun_bangun
Batas bawah: 1960.5
Batas atas : 2042.5

Shape akhir: (130, 7)
Dataset bersih tersimpan!


,id,luas_m2,harga_juta,kota,kamar,tahun_bangun,kondisi
0,1,297.0,1084.0,Jogja,2.0,2000.0,baik
1,2,254.0,761.0,Medan,1.0,1995.0,bagus
2,3,249.7,895.0,Depok,1.0,1983.0,baik
3,4,49.7,178.0,Ygy,5.0,2013.0,baik
4,5,133.4,424.0,Medan,5.0,2004.0,sedang


Dataset bersih berhasil disimpan sebagai housing_clean.csv


,id,name,email,address_city
0,1,Leanne Graham,Sincere@april.biz,Gwenborough
1,2,Ervin Howell,Shanna@melissa.tv,Wisokyburgh
2,3,Clementine Bauch,Nathan@yesenia.net,McKenziehaven
3,4,Patricia Lebsack,Julianne.OConner@kory.org,South Elvis
4,5,Chelsey Dietrich,Lucio_Hettinger@annie.ca,Roscoeview
5,6,Mrs. Dennis Schulist,Karley_Dach@jasper.info,South Christy
6,7,Kurtis Weissnat,Telly.Hoeger@billy.biz,Howemouth
7,8,Nicholas Runolfsdottir V,Sherwood@rosamond.me,Aliyaview
8,9,Glenna Reichert,Chaim_McDermott@dana.io,Bartholomebury
9,10,Clementina DuBuque,Rey.Padberg@karina.biz,Lebsackbury


Data API berhasil disimpan sebagai jsonplaceholder_users.csv
